# Precision of Quantum Fourier Transforms
Habit Blunk

### Introduction
I want to implement the quantum Fourier transform (QFT) in PennyLane and run several tests to determine behaviors such as unitarity, known states, against a discrete Fourier transform, its round trip performance, if it can detect periodicity, and how it handles fidelity scaling, to find out how it performs under noise and to what degree it will be off from exact.

In [ ]:
import pennylane as qp
import numpy as np

In [ ]:
# helper functions

def exact_dft_matrix(n_qubits: int) -> np.ndarray:
    n = 2 ** n_qubits
    omega = np.exp(2j * np.pi / n)
    return np.array([[omega ** (j * k) for k in range(n)] for j in range(n)]) / np.sqrt(n)

def fidelity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.abs(np.dot(np.conj(a), b)) ** 2)

def qft_statevector(n_qubits: int, input_state: np.ndarray) -> np.ndarray:
    dev = qp.device("default.qubit", wires=n_qubits)

    @qp.qnode(dev)
    def circuit():
        qp.StatePrep(input_state, wires=range(n_qubits))
        qp.QFT(wires=range(n_qubits))
        return qp.state()
    
    return np.array(circuit())
    
def iqft_statevector(n_qubits: int, input_state: np.ndarray) -> np.ndarray:
    dev = qp.device("default.qubit", wires=n_qubits)

    @qp.qnode(dev)
    def circuit():
        qp.StatePrep(input_state, wires=range(n_qubits))
        qp.adjoint(qp.QFT)(wires=range(n_qubits))
        return qp.state()

    return np.array(circuit())

In [ ]:
# test functions

def test_unitarity(n_qubits: int = 3, tol: float = 1e-6) -> bool:
    print(f"test 1 - unitarity (n_qubits={n_qubits})")

    n = 2 ** n_qubits
    u = np.zeros((n, n), dtype=complex)
    for k in range(n):
        basis = np.zeros(n)
        basis[k] = 1.0
        u[:, k] = qft_statevector(n_qubits, basis)

    product = u.conj().T @ u
    deviation = np.max(np.abs(product - np.eye(n)))
    passed = deviation < tol
    print(f"max |U†U - I| = {deviation:.2e}  →  {'pass ✓' if passed else 'fail ✗'}")
    return passed

def test_known_states(n_qubits: int = 3, tol: float = 1e-6) -> bool:
    print(f"\ntest 2 - known-state trnsforms (n_qubits={n_qubits})")
    n = 2 ** n_qubits
    all_passed = True

    # |0>
    zero = np.zeros(n)
    zero[0] = 1.0
    out = qft_statevector(n_qubits, zero)
    expected = np.ones(n) / np.sqrt(n)
    err = np.max(np.abs(out - expected))
    ok = err < tol
    all_passed &= ok
    print(f"QFT|0⟩ -> uniform superposition   max_err={err:.2e}  {'pass ✓' if ok else 'fail ✗'}")

    # |1>
    one = np.zeros(n)
    one[1] = 1.0
    out = qft_statevector(n_qubits, one)
    expected = np.array([np.exp(2j * np.pi * j / n) for j in range(n)]) / np.sqrt(n)
    err = np.max(np.abs(out - expected))
    ok = err < tol
    all_passed &= ok
    print(f"QFT|1⟩ -> linear phase ramp       max_err={err:.2e}  {'pass ✓' if ok else 'fail ✗'}")
    
    # |ψ>
    k = n // 3
    psi = np.zeros(n)
    psi[k] = 1.0
    out = qft_statevector(n_qubits, psi)
    expected = np.array([np.exp(2j * np.pi * j * k / n) for j in range(n)]) / np.sqrt(n)
    err = np.max(np.abs(out - expected))
    ok = err < tol
    all_passed &= ok
    print(f"QFT|ψ⟩ -> quadratic phase       max_err={err:.2e}  {'pass ✓' if ok else 'fail ✗'}")

    return all_passed

def test_vs_exact_dft(n_qubits: int = 4, tol: float = 1e-6) -> bool:
    print(f"\ntest 3 - qft vs exact dft matrix (n_qubits={n_qubits})")
    n = 2 ** n_qubits
    U_q = np.zeros((n, n), dtype=complex)
    for k in range(n):
        basis = np.zeros(n)
        basis[k] = 1.0
        U_q[:, k] = qft_statevector(n_qubits, basis)
    U_d = exact_dft_matrix(n_qubits)
    diff = np.max(np.abs(U_q - U_d))
    passed = diff < tol
    print(f"max element-wise |U_qft - U_dft| = {diff:.2e}  →  {'pass ✓' if passed else 'fail ✗'}")
    return passed

def test_round_trip(n_qubits: int = 4, n_trials: int = 5, tol: float = 1e-6) -> bool:
    print(f"\ntest 4 - round-trip qft -> iqft (n_qubits={n_qubits}, trials={n_trials})")
    rng = np.random.default_rng(42)
    n = 2 ** n_qubits
    all_passed = True
    
    for i in range(n_trials):
        raw = rng.standard_normal(n) + 1j * rng.standard_normal(n)
        psi = raw / np.linalg.norm(raw)
        recovered = iqft_statevector(n_qubits, qft_statevector(n_qubits, psi))
        fid = fidelity(psi, recovered)
        err = np.max(np.abs(psi - recovered))
        ok = (1 - fid) < tol and err < tol
        all_passed &= ok
        print(f"trial {i+1}: fidelity={fid:.8f}  max_err={err:.2e}  {'pass ✓' if ok else 'fail ✗'}")

    return all_passed

def test_periodicity_detection(n_qubits: int = 4, tol: float = 1e-4) -> bool:
    print(f"\ntest 5 - periodicity detection (n_qubits={n_qubits})")
    n = 2 ** n_qubits
    all_passed = True

    for period in [2, 4]:
        state = np.zeros(n, dtype=complex)
        indices = list(range(0, n, period))
        state[indices] = 1.0 / np.sqrt(len(indices))

        out = qft_statevector(n_qubits, state)
        probs = np.abs(out) ** 2

        step = n // period
        peak_positions = [k * step for k in range(period)]
        peak_prob = sum(probs[p] for p in peak_positions)
        ok = peak_prob > (1 - tol)
        all_passed &= ok
        print(f"period={period}  expected peaks={peak_positions}  "
              f"peak_prob={peak_prob:.6f}  {'pass ✓' if ok else 'fail ✗'}")
    
    return all_passed

def test_fidelity_scaling(max_qubits: int = 6) -> None:
    print(f"\ninfo: fidelity scaling from 2 to {max_qubits} qubits")
    print(f"  {'n_qubits':>8}  {'N':>6}  {'max_err':>12}  {'avg_err':>12}")
    for i in range(2, max_qubits + 1):
        n = 2 ** i
        U_q = np.zeros((n, n), dtype=complex)
        for k in range(n):
            basis = np.zeros(n)
            basis[k] = 1.0
            U_q[:, k] = qft_statevector(i, basis)
        U_exact = exact_dft_matrix(i)
        errs = np.abs(U_q - U_exact)
        print(f"{i:>8} {n:>6} {errs.max():>12.2e} {errs.mean():>12.2e}")

In [ ]:
results = {}
results["unitarity"] = test_unitarity(n_qubits=3)
results["known_states"] = test_known_states(n_qubits=3)
results["vs_exact_dft"] = test_vs_exact_dft(n_qubits=4)
results["round_trip"] = test_round_trip(n_qubits=4, n_trials=5)
results["periodicity"] = test_periodicity_detection(n_qubits=4)
test_fidelity_scaling(max_qubits=6)

print("summary")
all_ok = True
for name, passed in results.items():
    status = "pass ✓" if passed else "fail ✗"
    print(f" {name:<30} {status}")
    all_ok &= passed

print(f"overall: {'all tests passed ✓' if all_ok else 'some tests failed ✗'}")

### Discussion

Overall, this went as predicted. I was able to determine that, within an average tolerance of 0.0000001, the quantum Fourier transform is accurate in replicating unitarity, known states, compared to an exact discrete Fourier transform, round trip, and can detect periodicity. Its fidelity is also quite good.